# **4. Central Limit Theorem & Confidence Intervals**

## The Central Limit Theorem (CLT)

The **Central Limit Theorem** is one of the most important results in statistics:

> When you take sufficiently large random samples from ANY population, the distribution of sample means will be approximately normally distributed.

### Key Points
- Works regardless of the shape of the original distribution
- Sample size n ≥ 30 is usually sufficient
- Mean of sample means = population mean (μ)
- Standard error: $SE = \frac{\sigma}{\sqrt{n}}$

## Why CLT Matters
- Allows inference about populations using sample data
- Foundation for confidence intervals and hypothesis tests
- Explains why normal distribution appears everywhere

In [ ]:
# ===============================================
# CENTRAL LIMIT THEOREM DEMONSTRATION
# ===============================================

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

# Start with a clearly NON-normal population (uniform)
population = np.random.uniform(0, 100, size=100000)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Top row: population
axes[0, 0].hist(population, bins=50, color='steelblue', edgecolor='white')
axes[0, 0].set_title('Original Population\n(Uniform - NOT Normal!)')
axes[0, 0].axvline(np.mean(population), color='red', linewidth=2)

# Bottom row: sampling distributions for different n
sample_sizes = [5, 30, 100]
n_samples = 1000

for idx, n in enumerate(sample_sizes):
    sample_means = [np.mean(np.random.choice(population, size=n)) 
                    for _ in range(n_samples)]
    
    ax = axes[1, idx]
    ax.hist(sample_means, bins=40, density=True, color='coral', edgecolor='white', alpha=0.7)
    
    # Overlay normal curve
    x = np.linspace(min(sample_means), max(sample_means), 100)
    se = np.std(population) / np.sqrt(n)
    ax.plot(x, stats.norm.pdf(x, np.mean(population), se), 'b-', linewidth=2)
    
    ax.set_title(f'Sample Means (n={n})\nSE = σ/√n = {se:.2f}')
    ax.axvline(np.mean(sample_means), color='red', linewidth=2)

# Remove empty plots
axes[0, 1].axis('off')
axes[0, 2].axis('off')

plt.suptitle('Central Limit Theorem: Uniform → Normal!', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 Key Insight: As sample size increases, the distribution of")
print("   sample means becomes more normal and less spread out!")

In [ ]:
# ===============================================
# CLT WORKS FOR ANY DISTRIBUTION
# ===============================================

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

# Create different population types
populations = {
    'Exponential': np.random.exponential(5, 100000),
    'Bimodal': np.concatenate([np.random.normal(30, 5, 50000), 
                               np.random.normal(70, 5, 50000)]),
    'Skewed (Chi-sq)': np.random.chisquare(df=3, size=100000)
}

fig, axes = plt.subplots(len(populations), 2, figsize=(12, 10))

for idx, (name, pop) in enumerate(populations.items()):
    # Original distribution
    axes[idx, 0].hist(pop, bins=50, color='steelblue', edgecolor='white', density=True)
    axes[idx, 0].set_title(f'{name} Population')
    
    # Sampling distribution (n=50)
    n = 50
    sample_means = [np.mean(np.random.choice(pop, size=n)) for _ in range(1000)]
    
    axes[idx, 1].hist(sample_means, bins=40, color='coral', edgecolor='white', density=True)
    
    # Overlay normal
    x = np.linspace(min(sample_means), max(sample_means), 100)
    se = np.std(pop) / np.sqrt(n)
    axes[idx, 1].plot(x, stats.norm.pdf(x, np.mean(pop), se), 'b-', linewidth=2)
    axes[idx, 1].set_title(f'Distribution of Sample Means (n={n})')

plt.suptitle('CLT: Any Distribution → Normal Sample Means', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
# **Confidence Intervals**

A **confidence interval** provides a range of plausible values for a population parameter.

## Formula (for population mean)
$$\bar{x} \pm z^* \cdot \frac{\sigma}{\sqrt{n}}$$

or when σ is unknown (using sample std dev):
$$\bar{x} \pm t^* \cdot \frac{s}{\sqrt{n}}$$

## Common Z* Values
| Confidence Level | Z* |
|-----------------|----|
| 90% | 1.645 |
| 95% | 1.96 |
| 99% | 2.576 |

## What Does "95% Confident" Mean?
If we repeated the sampling process many times, 95% of the intervals would contain the true population mean.

In [ ]:
# ===============================================
# CALCULATING CONFIDENCE INTERVALS
# ===============================================

import numpy as np
from scipy import stats

print("=" * 50)
print("CONFIDENCE INTERVAL CALCULATION")
print("=" * 50)

# Sample data: heights of 36 students
np.random.seed(42)
sample = np.random.normal(170, 10, 36)  # True μ=170, σ=10

n = len(sample)
x_bar = np.mean(sample)
s = np.std(sample, ddof=1)
se = s / np.sqrt(n)

print(f"\nSample Statistics:")
print(f"  n = {n}")
print(f"  x̄ = {x_bar:.2f}")
print(f"  s = {s:.2f}")
print(f"  SE = s/√n = {se:.2f}")

# 95% confidence interval
confidence = 0.95
alpha = 1 - confidence

# Using t-distribution (unknown σ)
t_star = stats.t.ppf(1 - alpha/2, df=n-1)
margin_error = t_star * se

ci_lower = x_bar - margin_error
ci_upper = x_bar + margin_error

print(f"\n95% Confidence Interval:")
print(f"  t* = {t_star:.3f}")
print(f"  Margin of Error = t* × SE = {margin_error:.2f}")
print(f"  CI = [{ci_lower:.2f}, {ci_upper:.2f}]")

# Using scipy directly
ci_scipy = stats.t.interval(confidence, df=n-1, loc=x_bar, scale=se)
print(f"\nUsing scipy: [{ci_scipy[0]:.2f}, {ci_scipy[1]:.2f}]")

print(f"\n✓ True population mean (170) is {'INSIDE' if ci_lower <= 170 <= ci_upper else 'OUTSIDE'} the interval!")

In [ ]:
# ===============================================
# VISUALIZING CONFIDENCE INTERVAL MEANING
# ===============================================

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

# True population
true_mean = 100
true_std = 15
population = np.random.normal(true_mean, true_std, 10000)

# Take 50 samples and compute 95% CI for each
n_experiments = 50
sample_size = 30
confidence = 0.95

results = []
for i in range(n_experiments):
    sample = np.random.choice(population, size=sample_size)
    x_bar = np.mean(sample)
    se = np.std(sample, ddof=1) / np.sqrt(sample_size)
    ci = stats.t.interval(confidence, df=sample_size-1, loc=x_bar, scale=se)
    contains_true = ci[0] <= true_mean <= ci[1]
    results.append((x_bar, ci[0], ci[1], contains_true))

# Visualization
fig, ax = plt.subplots(figsize=(12, 10))

for i, (x_bar, lower, upper, contains) in enumerate(results):
    color = 'green' if contains else 'red'
    ax.plot([lower, upper], [i, i], color=color, linewidth=2)
    ax.scatter([x_bar], [i], color=color, s=30)

ax.axvline(x=true_mean, color='blue', linewidth=2, linestyle='--', 
           label=f'True μ = {true_mean}')

# Count successes
n_contains = sum(r[3] for r in results)

ax.set_xlabel('Value', fontsize=12)
ax.set_ylabel('Sample #', fontsize=12)
ax.set_title(f'95% Confidence Intervals from {n_experiments} Samples\n'
             f'{n_contains}/{n_experiments} ({n_contains/n_experiments:.0%}) contain true mean',
             fontsize=14)
ax.legend(loc='upper right')

plt.tight_layout()
plt.show()

print("💡 About 95% of intervals contain the true mean.")
print("   Green = contains true mean, Red = misses it")

In [ ]:
# ===============================================
# FACTORS AFFECTING CONFIDENCE INTERVAL WIDTH
# ===============================================

import numpy as np
from scipy import stats

print("=" * 50)
print("FACTORS AFFECTING CI WIDTH")
print("=" * 50)

x_bar = 100  # Fixed sample mean
s = 15       # Fixed sample std dev

# Factor 1: Sample size
print("\n1. Sample Size (95% CI, s=15):")
for n in [10, 30, 100, 500]:
    se = s / np.sqrt(n)
    ci = stats.t.interval(0.95, df=n-1, loc=x_bar, scale=se)
    width = ci[1] - ci[0]
    print(f"   n={n:3d}: [{ci[0]:.1f}, {ci[1]:.1f}] (width = {width:.1f})")

# Factor 2: Confidence level
print("\n2. Confidence Level (n=30, s=15):")
n = 30
se = s / np.sqrt(n)
for conf in [0.80, 0.90, 0.95, 0.99]:
    ci = stats.t.interval(conf, df=n-1, loc=x_bar, scale=se)
    width = ci[1] - ci[0]
    print(f"   {conf:.0%} CI: [{ci[0]:.1f}, {ci[1]:.1f}] (width = {width:.1f})")

print("\n💡 Key Insights:")
print("   - Larger n → Narrower CI (more precision)")
print("   - Higher confidence → Wider CI (more certain)")
print("   - There's a tradeoff between precision and confidence!")

---
## Summary

| Concept | Key Formula/Insight |
|---------|--------------------|
| **CLT** | Sample means → Normal distribution |
| **Standard Error** | $SE = \frac{\sigma}{\sqrt{n}}$ |
| **95% CI** | $\bar{x} \pm 1.96 \cdot SE$ (or t* for small n) |
| **Interpretation** | 95% of CIs will contain true μ |
| **Narrower CI** | Increase n or decrease confidence |